In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", torch_dtype=DTYPE).to(DEVICE)
model.eval()

PROMPT = "Write one sentence explaining what entropy means in information theory."

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
print(model)
print("\n--- Top-level structure ---")
for name, child in model.named_children():
    print(f"  {name}: {type(child).__name__}")

base = getattr(model, "model", model)
embed_module = getattr(base, "embed_tokens", None) or getattr(base, "embed", None)
layers_list = getattr(base, "layers", None)
final_norm = getattr(base, "norm", None)
lm_head_module = getattr(model, "lm_head", None)

assert embed_module is not None, "embed_tokens/embed not found"
assert layers_list is not None, "layers not found"
assert final_norm is not None, "norm not found"
assert lm_head_module is not None, "lm_head not found"

num_blocks = len(layers_list)
print(f"\nembed_tokens: {type(embed_module).__name__}")
print(f"layers: {num_blocks} blocks")
print(f"final norm: {type(final_norm).__name__}")
print(f"lm_head: {type(lm_head_module).__name__}")

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [5]:
inputs = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
input_ids_baseline = inputs["input_ids"].clone()
num_new_tokens = 16

with torch.no_grad():
    for _ in range(num_new_tokens):
        outputs = model(input_ids=input_ids_baseline, use_cache=False)
        logits = outputs.logits
        next_token_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        input_ids_baseline = torch.cat([input_ids_baseline, next_token_id], dim=-1)

generated_ids_baseline = input_ids_baseline[0, inputs["input_ids"].shape[1]:].tolist()
decoded_baseline = tokenizer.decode(generated_ids_baseline, skip_special_tokens=True)
print("Baseline generated token IDs:", generated_ids_baseline)
print("Baseline decoded:", decoded_baseline)

Baseline generated token IDs: [4863, 17764, 374, 264, 6629, 315, 279, 19267, 476, 86690, 304, 264, 1849, 13, 758, 1995]
Baseline decoded:  Entropy is a measure of the disorder or randomness in a system. In information


In [6]:
def run_middle_stage(hidden_states):
    seq_len = hidden_states.shape[1]
    position_ids = torch.arange(seq_len, device=DEVICE, dtype=torch.long).unsqueeze(0)
    position_embeddings = model.model.rotary_emb(hidden_states, position_ids)
    causal_mask = torch.triu(
        torch.full((seq_len, seq_len), torch.finfo(hidden_states.dtype).min, device=DEVICE, dtype=hidden_states.dtype),
        diagonal=1,
    ).unsqueeze(0).unsqueeze(0)
    for layer in layers_list:
        hidden_states = layer(
            hidden_states,
            attention_mask=causal_mask,
            position_embeddings=position_embeddings,
            position_ids=position_ids,
            use_cache=False,
        )
    return hidden_states

input_ids_split = inputs["input_ids"].clone()
with torch.no_grad():
    for _ in range(num_new_tokens):
        hidden_local = embed_module(input_ids_split)
        hidden_middle = run_middle_stage(hidden_local)
        hidden_after_norm = final_norm(hidden_middle)
        logits_split = lm_head_module(hidden_after_norm[:, -1:, :])
        next_token_id = logits_split.argmax(dim=-1)
        input_ids_split = torch.cat([input_ids_split, next_token_id], dim=-1)

generated_ids_split = input_ids_split[0, inputs["input_ids"].shape[1]:].tolist()
decoded_split = tokenizer.decode(generated_ids_split, skip_special_tokens=True)
print("Split generated token IDs:", generated_ids_split)
print("Split decoded:", decoded_split)

Split generated token IDs: [4863, 17764, 374, 264, 6629, 315, 279, 19267, 476, 86690, 304, 264, 1849, 13, 758, 1995]
Split decoded:  Entropy is a measure of the disorder or randomness in a system. In information


In [7]:
match = generated_ids_baseline == generated_ids_split
print("Token-by-token comparison:")
for i in range(num_new_tokens):
    status = "ok" if generated_ids_baseline[i] == generated_ids_split[i] else "MISMATCH"
    print(f"  {i}: baseline={generated_ids_baseline[i]}, split={generated_ids_split[i]} [{status}]")
print("\nPASS" if match else "FAIL")
print("\nBaseline decoded:", decoded_baseline)
print("Split decoded:    ", decoded_split)

Token-by-token comparison:
  0: baseline=4863, split=4863 [ok]
  1: baseline=17764, split=17764 [ok]
  2: baseline=374, split=374 [ok]
  3: baseline=264, split=264 [ok]
  4: baseline=6629, split=6629 [ok]
  5: baseline=315, split=315 [ok]
  6: baseline=279, split=279 [ok]
  7: baseline=19267, split=19267 [ok]
  8: baseline=476, split=476 [ok]
  9: baseline=86690, split=86690 [ok]
  10: baseline=304, split=304 [ok]
  11: baseline=264, split=264 [ok]
  12: baseline=1849, split=1849 [ok]
  13: baseline=13, split=13 [ok]
  14: baseline=758, split=758 [ok]
  15: baseline=1995, split=1995 [ok]

PASS

Baseline decoded:  Entropy is a measure of the disorder or randomness in a system. In information
Split decoded:      Entropy is a measure of the disorder or randomness in a system. In information
